In [26]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [27]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [28]:
df = pd.read_csv("IMDB Dataset.csv")
df.shape
df.head()
df.isnull().sum()
df.drop_duplicates(inplace = True)
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


# data preprocessing

In [29]:
###- converting to lower case values
df["review"] = df["review"].str.lower()

###- removing the URL's
import re #regularization expression
sample_text = "abc is word , abc" # abc --> xyz
new_text = re.sub("abc", "xyz", sample_text)

### removing links
def remove_urls(text):
    text = re.sub(r"https\S+", "",text) #pattern define, replacement, string
    return text
df["review"] = df["review"].apply(remove_urls)

### removing punctuations

def remove_punc(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) ## A-Z, a-z, 0-9 \s
    return text
df["review"] = df["review"].apply(remove_punc)

### removing html tags

def remove_html(text):
    text = re.sub(r"<.*?>", " ", text) # removing tags
    return text
df["review"] = df["review"].apply(remove_html)

# removing the stop words

In [30]:
import nltk # natural language tool kit
nltk.download("punkt") # tokenizer in nltk
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/kshitiznagar/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kshitiznagar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kshitiznagar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [31]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stop_words(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text.replace(word, "")
    return text

df["review"] = df["review"].apply(remove_stop_words)

In [32]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


# stemming

In [34]:
# running -> run like that
# we are using porter stemming
from nltk.stem import PorterStemmer

In [37]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_word = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_word.append(stemmed_token)
    return " ".join(stemmed_word)

df["review"] = df["review"].apply(stemming)

In [38]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the film techniqu...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love in the time of money is a v...,positive


# Encoding 

In [39]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

### vectorization

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)
X = tf.fit_transform(df["review"])
y = df["sentiment"]

# Dataset and Dataloader

In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train = X_train.toarray()
X_test = X_test.toarray()

In [52]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float(),
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float(),
)

In [53]:
train_loader = DataLoader(train_set, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_set, batch_size = 64, shuffle = True)

# RNN

In [56]:
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 128, num_layers = 1):
        super(RNN, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)#optinal => shape (num_of_layers, batch_size, hidden_size)
        out, _ = self.rnn(x, h0) 
        # returns two values 1st value is hidden state of all the timesteps [batch, sq_len, hidden_size]
        # 2nd value is final hidden state of last timesteps

        out = self.fc(out[:, -1, :])
        return out

In [60]:
imput_size = X_train.shape[1]
model = RNN(imput_size)
my_model = model

criterion = nn.BCELoss()
optimizer = optim.Adam(my_model.parameters())

# training the RNN

In [62]:
epochs = 10

for epoch in range(epochs):
    my_model.train()

    for xb, yb in train_loader:
        xb = xb.unsqueeze(1)
        optimizer.zero_grad()

        outputs = my_model(xb)
        final_outputs = torch.sigmoid(outputs.squeeze()) # probability
        loss = criterion(final_outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch is {epoch + 1} loss is {loss.item()}")

epoch is 1 loss is 0.3285435140132904
epoch is 2 loss is 0.24199524521827698
epoch is 3 loss is 0.18631984293460846
epoch is 4 loss is 0.26828858256340027
epoch is 5 loss is 0.20646198093891144
epoch is 6 loss is 0.16864025592803955
epoch is 7 loss is 0.10316947847604752
epoch is 8 loss is 0.09380495548248291
epoch is 9 loss is 0.17521615326404572
epoch is 10 loss is 0.25765562057495117


# evaluate

In [64]:
my_model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        outputs = my_model(xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()
    print(f"Accuracy is {correct_vals/total_vals*100}")

Accuracy is 87.32479580518302
